# Playground S6E7 — Cross-submission blender (LOCAL notebook)

Ensembles **across prior runs' submissions** by blending their saved class-probability
artifacts, exactly the mechanism behind the public LB-0.95075 notebook
([vad13irt](https://www.kaggle.com/code/vad13irt/ps-s6e7-eda-ensemble-lb-0-95075)) — whose
"ensemble" is literally `0.6*subA + 0.4*subB` over two public submissions' probability files.

**How it plugs into the pipeline:**
- Each `ply-s6e7-ft.ipynb` run writes `test_proba.csv` + `oof_proba.csv` (with `y_true`)
  next to `submission.csv`; `scripts/collect_run.py` archives them to
  `experiments/preds/<run_id>/` and records the path in `experiments/runs.csv` (`preds_dir`).
- This notebook loads every archived run, **hill-climbs blend weights on OOF balanced
  accuracy**, re-runs the per-class decision-weight search on the blended OOF, and writes a
  blended `submission.csv` + manifest under `experiments/blends/<timestamp>/`, appending a
  row to `runs.csv`.

**Rules of the road:**
- This notebook runs **locally from the repo root** (no `/kaggle/input`). It is *not* the
  kernel pushed to Kaggle — `kernel-metadata.json` must keep pointing at `ply-s6e7-ft.ipynb`.
- Submit the blended file directly:
  `kaggle competitions submit playground-series-s6e7 -f experiments/blends/<stamp>/submission.csv -m "blend ..."`
  then fill `public_lb_score` into the blend's `runs.csv` row by hand.
- **Overfit caution (v6 lesson):** an OOF gain from reweighting near-identical runs may not
  transfer to the LB — our runs share one pipeline, so their errors are highly correlated.
  Blending helps most when runs genuinely differ (feature sets, learners, seeds). Keep the
  degrees of freedom low and treat small OOF wins (< ~5e-4) as noise.

In [ ]:
import csv, json, uuid
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score

REPO_ROOT = Path.cwd()                      # run from the repo root
assert (REPO_ROOT / "kernel-metadata.json").exists(), "run this notebook from the repo root"
PREDS_DIR = REPO_ROOT / "experiments" / "preds"
BLENDS_DIR = REPO_ROOT / "experiments" / "blends"
RUNS_CSV = REPO_ROOT / "experiments" / "runs.csv"

APPEND_TO_RUNS_CSV = True                   # log the blend as a run row
WEIGHT_MARGIN = 5e-4                        # same guard as the main notebook

runs_log = pd.read_csv(RUNS_CSV)

run_dirs = {}
if PREDS_DIR.exists():
    for d in sorted(PREDS_DIR.iterdir()):
        if (d / "oof_proba.csv").exists() and (d / "test_proba.csv").exists():
            run_dirs[d.name] = d
print(f"Archived runs with probability artifacts: {list(run_dirs)}")
if len(run_dirs) < 2:
    print("NOTE: fewer than 2 runs archived -- the blend degenerates to the best single run. "
          "Collect more runs via scripts/collect_run.py first.")

# context from the experiment log, where available
log_info = runs_log.set_index("run_id")[["description", "final_oof_bal_acc", "public_lb_score"]]
for rid in run_dirs:
    if rid in log_info.index:
        print(f"  {rid}: {log_info.loc[rid, 'description']}  "
              f"(OOF {log_info.loc[rid, 'final_oof_bal_acc']}, LB {log_info.loc[rid, 'public_lb_score']})")

## Load and align the artifacts

All runs are aligned by `id` (fold seeds may differ across runs — irrelevant, each OOF file
covers the full train set). `y_true` must agree exactly across runs; class columns are taken
from the first run and asserted identical everywhere.

In [ ]:
CLASSES, y, oof_ids, test_ids = None, None, None, None
names, oof_mats, test_mats = [], [], []

for rid, d in run_dirs.items():
    oof = pd.read_csv(d / "oof_proba.csv").sort_values("id").reset_index(drop=True)
    tst = pd.read_csv(d / "test_proba.csv").sort_values("id").reset_index(drop=True)
    if CLASSES is None:
        CLASSES = [c for c in oof.columns if c not in ("id", "y_true")]
        cls_to_int = {c: k for k, c in enumerate(CLASSES)}
        oof_ids, test_ids = oof["id"].values, tst["id"].values
        y = oof["y_true"].map(cls_to_int).values
    else:
        assert [c for c in oof.columns if c not in ("id", "y_true")] == CLASSES, f"{rid}: class column mismatch"
        assert np.array_equal(oof["id"].values, oof_ids) and np.array_equal(tst["id"].values, test_ids), f"{rid}: id mismatch"
        assert np.array_equal(oof["y_true"].map(cls_to_int).values, y), f"{rid}: y_true mismatch"
    names.append(rid)
    oof_mats.append(oof[CLASSES].values)
    test_mats.append(tst[CLASSES].values)

P_oof, P_test = np.stack(oof_mats), np.stack(test_mats)   # (n_runs, n, K)
print(f"P_oof {P_oof.shape}  P_test {P_test.shape}  classes {CLASSES}")

single_scores = {n: balanced_accuracy_score(y, P_oof[i].argmax(1)) for i, n in enumerate(names)}
for n, s in sorted(single_scores.items(), key=lambda kv: -kv[1]):
    print(f"  {n}: OOF bal_acc {s:.5f}")

## Hill-climb blend weights on OOF balanced accuracy

Classic Kaggle hill climbing *with repetition*: start from the best single run, then
repeatedly add whichever run (repeats allowed) most improves the running average — weights
are implied by selection counts, so the search has few degrees of freedom and can't produce
wild weights. Compared against the simple average; best OOF wins. Then the per-class
decision-weight search from the main notebook runs on the blended OOF (same
`WEIGHT_MARGIN` guard against adopting noise).

In [ ]:
def hill_climb(P, y, n_steps=60, tol=1e-6):
    n_runs = P.shape[0]
    singles = [balanced_accuracy_score(y, P[i].argmax(1)) for i in range(n_runs)]
    counts = np.zeros(n_runs)
    counts[int(np.argmax(singles))] = 1
    best = max(singles)
    for _ in range(n_steps):
        cand = []
        for i in range(n_runs):
            c_try = counts.copy(); c_try[i] += 1
            blend = np.tensordot(c_try / c_try.sum(), P, axes=1)
            cand.append(balanced_accuracy_score(y, blend.argmax(1)))
        i_best = int(np.argmax(cand))
        if cand[i_best] <= best + tol:
            break
        counts[i_best] += 1
        best = cand[i_best]
    return counts / counts.sum(), best


def optimize_class_weights(y_true, proba, n_rounds=3, grid=np.linspace(0.75, 1.35, 25)):
    """Coordinate-ascent per-class decision weights (copied from ply-s6e7-ft.ipynb)."""
    w = np.ones(proba.shape[1])
    best_score = balanced_accuracy_score(y_true, proba.argmax(1))
    for _ in range(n_rounds):
        improved = False
        for k in range(proba.shape[1]):
            best_wk, best_local = w[k], best_score
            for cand in grid:
                w_try = w.copy(); w_try[k] = cand
                s = balanced_accuracy_score(y_true, (proba * w_try).argmax(1))
                if s > best_local:
                    best_local, best_wk = s, cand
            if best_local > best_score:
                w[k] = best_wk; best_score = best_local; improved = True
        if not improved:
            break
    return w, best_score


w_hill, ba_hill = hill_climb(P_oof, y)
w_avg = np.full(len(names), 1.0 / len(names))
ba_avg = balanced_accuracy_score(y, np.tensordot(w_avg, P_oof, axes=1).argmax(1))
print(f"hill-climb OOF {ba_hill:.5f}  weights {dict(zip(names, w_hill.round(3)))}")
print(f"simple-avg OOF {ba_avg:.5f}")

w_runs, blend_method = (w_hill, "hill_climb") if ba_hill >= ba_avg else (w_avg, "simple_avg")
blend_oof  = np.tensordot(w_runs, P_oof,  axes=1)
blend_test = np.tensordot(w_runs, P_test, axes=1)
ba_blend = balanced_accuracy_score(y, blend_oof.argmax(1))

w_cls, ba_cls = optimize_class_weights(y, blend_oof)
if ba_cls >= ba_blend + WEIGHT_MARGIN:
    class_weights, class_weight_method, final_oof = w_cls, "search", ba_cls
else:
    class_weights, class_weight_method, final_oof = np.ones(len(CLASSES)), "none", ba_blend

print(f"\nblend={blend_method} OOF {ba_blend:.5f}  "
      f"+class-weights({class_weight_method}) -> {final_oof:.5f}  w={class_weights.round(3).tolist()}")
best_single = max(single_scores.values())
print(f"best single run OOF {best_single:.5f}  ->  blend delta {final_oof - best_single:+.5f}"
      + ("   (below noise, expect no LB lift)" if final_oof - best_single < 5e-4 else ""))

## Write the blended submission + manifest, log the run

Outputs land in `experiments/blends/<UTC stamp>/` (gitignored). A row is appended to
`experiments/runs.csv` with the blend recipe in `notes`; fill in `public_lb_score` after
submitting.

In [ ]:
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
out_dir = BLENDS_DIR / stamp
out_dir.mkdir(parents=True, exist_ok=True)

final_pred = (blend_test * class_weights).argmax(1)
sub = pd.DataFrame({"id": test_ids, "health_condition": [CLASSES[i] for i in final_pred]})
sub.to_csv(out_dir / "submission.csv", index=False)

# blended probabilities are themselves a blendable artifact for future rounds
tp = pd.DataFrame({"id": test_ids}); op = pd.DataFrame({"id": oof_ids, "y_true": [CLASSES[i] for i in y]})
for k, c in enumerate(CLASSES):
    tp[c] = blend_test[:, k].round(6); op[c] = blend_oof[:, k].round(6)
tp.to_csv(out_dir / "test_proba.csv", index=False)
op.to_csv(out_dir / "oof_proba.csv", index=False)

manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "runs": names,
    "run_weights": dict(zip(names, np.round(w_runs, 4).tolist())),
    "blend_method": blend_method,
    "class_weight_method": class_weight_method,
    "class_weights": np.round(class_weights, 4).tolist(),
    "single_oof": {n: round(float(s), 5) for n, s in single_scores.items()},
    "blend_oof_bal_acc": round(float(ba_blend), 5),
    "final_oof_bal_acc": round(float(final_oof), 5),
}
(out_dir / "blend_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(json.dumps(manifest, indent=2))
print(f"\nWrote {out_dir / 'submission.csv'}")
print(f"Predicted class balance:\n{sub['health_condition'].value_counts(normalize=True)}")

if APPEND_TO_RUNS_CSV:
    with RUNS_CSV.open(newline="", encoding="utf-8") as f:
        fieldnames = csv.DictReader(f).fieldnames
    row = {k: "" for k in fieldnames}
    row.update({
        "run_id": uuid.uuid4().hex[:8],
        "timestamp_utc": manifest["created_utc"],
        "description": f"blend[{blend_method}] of {'+'.join(names)}",
        "class_weight_method": class_weight_method,
        "final_oof_bal_acc": manifest["final_oof_bal_acc"],
        "notes": "weights " + json.dumps(manifest["run_weights"]) + "; fill public_lb_score after submitting",
        "preds_dir": out_dir.relative_to(REPO_ROOT).as_posix(),
    })
    for i, k in enumerate(("class_weight_w0", "class_weight_w1", "class_weight_w2")):
        if k in row and i < len(class_weights):
            row[k] = round(float(class_weights[i]), 4)
    with RUNS_CSV.open("a", newline="", encoding="utf-8") as f:
        csv.DictWriter(f, fieldnames=fieldnames).writerow(row)
    print(f"\nAppended blend run {row['run_id']} to {RUNS_CSV}")

print(f"\nSubmit with:\n  kaggle competitions submit playground-series-s6e7 "
      f"-f {out_dir / 'submission.csv'} -m \"{row['description'] if APPEND_TO_RUNS_CSV else 'blend'}\"")

## Appendix: label-only majority vote over legacy submissions

Runs from before probability artifacts existed only left hard labels
(`.kaggle_output/*/submission.csv`). A weighted label vote is strictly weaker than
probability blending (no confidence information) but lets those runs participate. Fill in
paths and weights (e.g. public LB scores) to use; leave empty to skip.

In [ ]:
LEGACY_SUBS = {
    # r".kaggle_output/20260710T155129Z/submission.csv": 0.94951,   # v5
    # r".kaggle_output/20260710T185815Z/submission.csv": 0.94944,   # v6
    # r".kaggle_output/20260710T232243Z/submission.csv": 0.94934,   # v7
}

if LEGACY_SUBS:
    votes = None
    for path, wgt in LEGACY_SUBS.items():
        s = pd.read_csv(REPO_ROOT / path).sort_values("id").reset_index(drop=True)
        onehot = pd.get_dummies(s["health_condition"]).reindex(columns=CLASSES, fill_value=0)
        votes = onehot * wgt if votes is None else votes + onehot * wgt
        vote_ids = s["id"].values
    vote_pred = [CLASSES[i] for i in votes.values.argmax(1)]
    vote_sub = pd.DataFrame({"id": vote_ids, "health_condition": vote_pred})
    vote_path = out_dir / "submission_labelvote.csv"
    vote_sub.to_csv(vote_path, index=False)
    print(f"Wrote {vote_path}")
    print(vote_sub["health_condition"].value_counts(normalize=True))
else:
    print("LEGACY_SUBS empty -- skipped.")